<a href="https://colab.research.google.com/github/smosharof/Resume.Walkthrough/blob/main/A_B_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import kagglehub
import pandas as pd
import os
from scipy import stats

try:
    path = kagglehub.dataset_download("amirmotefaker/ab-testing-dataset")
    print("Path to downloaded directory:", path)

    all_files = os.listdir(path)
    csv_files = [f for f in all_files if f.endswith('.csv')]

    if "control_group.csv" in csv_files and "test_group.csv" in csv_files:
        # Load the datasets from the downloaded directory
        df_control = pd.read_csv(os.path.join(path, "control_group.csv"), delimiter=";")
        df_test = pd.read_csv(os.path.join(path, "test_group.csv"), delimiter=";")

        # 1. Data Cleaning/Preprocessing
        # Handle missing values (for simplicity, fill NaN with 0)
        df_control = df_control.fillna(0)

        # Ensure consistent data types (convert all relevant columns to numeric)
        cols_to_convert = ['# of Impressions', 'Reach', '# of Website Clicks', '# of Searches',
                           '# of View Content', '# of Add to Cart', '# of Purchase']
        for col in cols_to_convert:
            df_control[col] = pd.to_numeric(df_control[col], errors='coerce')
            df_test[col] = pd.to_numeric(df_test[col], errors='coerce')

        # Convert Date to datetime format
        df_control['Date'] = pd.to_datetime(df_control['Date'], format='%d.%m.%Y')
        df_test['Date'] = pd.to_datetime(df_test['Date'], format='%d.%m.%Y')

        # 2. Metric Calculation
        # Calculate Click-Through Rate (CTR)
        df_control['CTR'] = (df_control['# of Website Clicks'] / df_control['# of Impressions']) * 100
        df_test['CTR'] = (df_test['# of Website Clicks'] / df_test['# of Impressions']) * 100

        # Calculate Add-to-Cart Rate
        df_control['Add_to_Cart_Rate'] = (df_control['# of Add to Cart'] / df_control['# of Website Clicks']) * 100
        df_test['Add_to_Cart_Rate'] = (df_test['# of Add to Cart'] / df_test['# of Website Clicks']) * 100

        # Calculate Purchase Rate (Conversion Rate)
        df_control['Purchase_Rate'] = (df_control['# of Purchase'] / df_control['# of Website Clicks']) * 100
        df_test['Purchase_Rate'] = (df_test['# of Purchase'] / df_test['# of Website Clicks']) * 100

        # Calculate Cost Per Click (CPC)
        df_control['CPC'] = df_control['Spend [USD]'] / df_control['# of Website Clicks']
        df_test['CPC'] = df_test['Spend [USD]'] / df_test['# of Website Clicks']

        # 3. Aggregation and Comparison
        # Aggregate metrics by Campaign Name (overall performance)
        agg_control = df_control.groupby('Campaign Name').mean().reset_index()
        agg_test = df_test.groupby('Campaign Name').mean().reset_index()

        # Combine aggregated data for easy comparison
        comparison_df = pd.concat([agg_control, agg_test]).set_index('Campaign Name')
        print("\nComparison of Overall Metrics:\n")
        print(comparison_df.to_markdown(numalign="left", stralign="left"))

        # Aggregate metrics by Date (trend analysis)
        agg_control_date = df_control.groupby('Date').sum().reset_index()
        agg_test_date = df_test.groupby('Date').sum().reset_index()

        # Combine for trend analysis
        trend_df = pd.merge(agg_control_date, agg_test_date, on='Date', suffixes=('_control', '_test'))
        print("\nTrend Analysis Data:\n")
        print(trend_df.head().to_markdown(index=False, numalign="left", stralign="left"))


        # 4. Statistical Significance Testing (Example - T-test for Purchase Rate)

        t_statistic, p_value = stats.ttest_ind(df_control['Purchase_Rate'].dropna(), df_test['Purchase_Rate'].dropna())
        print(f"\n\nT-test for Purchase Rate: t-statistic = {t_statistic:.4f}, p-value = {p_value:.4f}")
        if p_value < 0.05:
            print("The difference in Purchase Rate is statistically significant (p < 0.05)")
        else:
            print("The difference in Purchase Rate is not statistically significant (p >= 0.05)")


    else:
        print("Error: control_group.csv or test_group.csv not found in the downloaded dataset.")


except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure you have the kagglehub library installed.")

Path to downloaded directory: /kaggle/input/ab-testing-dataset

Comparison of Overall Metrics:

| Campaign Name    | Date                | Spend [USD]   | # of Impressions   | Reach   | # of Website Clicks   | # of Searches   | # of View Content   | # of Add to Cart   | # of Purchase   | CTR     | Add_to_Cart_Rate   | Purchase_Rate   | CPC      |
|:-----------------|:--------------------|:--------------|:-------------------|:--------|:----------------------|:----------------|:--------------------|:-------------------|:----------------|:--------|:-------------------|:----------------|:---------|
| Control Campaign | 2019-08-15 12:00:00 | 2288.43       | 105908             | 85883.4 | 5143.43               | 2147.27         | 1879                | 1256.67            | 505.367         | 5.09587 | 27.8209            | 11.4772         | inf      |
| Test Campaign    | 2019-08-15 12:00:00 | 2563.07       | 74584.8            | 53491.6 | 6032.33               | 2418.97         | 1858         